In [1]:
!pip install tensorflow pandas numpy

In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

In [6]:
ALPHABET = "abcdefghijklmnopqrstuvwxyz "
VOCAB_SIZE = len(ALPHABET)
char2idx = {c: i for i, c in enumerate(ALPHABET)}
idx2char = list(ALPHABET)

In [7]:
URL = "https://github.com/jpospinalo/MachineLearning/raw/main/nlp/dinos.csv"
df = pd.read_csv(URL, header=None, names=["name"])
names = df["name"].astype(str).str.lower().str.strip().tolist()
raw_text = " ".join(names)
clean_text = "".join([c if c in ALPHABET else " " for c in raw_text])
clean_text = " ".join(clean_text.split())

In [8]:
SEQ_LEN = 40
sequences, next_chars = [], []
for i in range(0, len(clean_text) - SEQ_LEN):
    seq = clean_text[i:i+SEQ_LEN]
    nxt = clean_text[i+SEQ_LEN]
    sequences.append([char2idx[c] for c in seq])
    next_chars.append(char2idx[nxt])

X = np.array(sequences, dtype=np.int32)
y = np.array(next_chars, dtype=np.int32)

In [13]:
MBED_DIM = 32
LSTM_UNITS = 256

model = models.Sequential([
    layers.Embedding(VOCAB_SIZE, MBED_DIM),
    layers.LSTM(LSTM_UNITS),
    layers.Dense(VOCAB_SIZE, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=['accuracy'])

In [10]:
model.fit(X, y, epochs=20, batch_size=256)

Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.1424 - loss: 2.9389
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3330 - loss: 2.3621
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3976 - loss: 2.0403
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4168 - loss: 1.9211
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4413 - loss: 1.8375
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4505 - loss: 1.7914
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4641 - loss: 1.7500
Epoch 8/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4852 - loss: 1.7051
Epoch 9/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5010 - loss: 1.6688
Epoch 10/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5195 - loss: 1.6233
Epoch 11/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5401 - loss: 1.5577
Epoch 12/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy:

```markdown
To generate new dinosaur names, I'll define a function that uses the trained model. The function will take an initial sequence of characters and then iteratively predict the next character, feeding the newly predicted character back into the input sequence until a full name is generated. We can control the 'creativity' or randomness of the generation using a `temperature` parameter.
```

In [12]:
def generate_name(model, start_string, num_generate=100, temperature=1.5):
    # Convert start string to numerical representation
    input_sequence = [char2idx[s] for s in start_string] # This list will be updated

    # Empty list to store generated characters
    text_generated = []

    for _ in range(num_generate):
        # Create a batch-like tensor from the current sequence
        input_eval = tf.expand_dims(input_sequence, 0) # Shape (1, SEQ_LEN)

        predictions = model(input_eval) # Output shape (1, VOCAB_SIZE)

        # Scale predictions by temperature
        predictions = predictions / temperature

        # Sample the next character from the categorical distribution
        # tf.random.categorical expects logits of shape [batch_size, num_classes]
        # Our predictions are already (1, VOCAB_SIZE), so this is correct.
        predicted_id = tf.random.categorical(predictions, num_samples=1)[0, 0].numpy()

        # Add the predicted character to the output
        text_generated.append(idx2char[predicted_id])

        # Update the input sequence: remove the first character and add the new one
        input_sequence.append(predicted_id)
        input_sequence = input_sequence[1:] # Maintain SEQ_LEN

    return start_string + ''.join(text_generated)

print("Generated names:")
for _ in range(10):
    # Start with a random short sequence from the training data, or a common prefix
    start_index = np.random.randint(0, len(clean_text) - SEQ_LEN)
    seed_text = clean_text[start_index : start_index + SEQ_LEN]
    generated = generate_name(model, seed_text, num_generate=20, temperature=0.5)
    # Clean up generated name by taking only up to the first space or a reasonable length
    clean_generated = generated.split(' ')[0]
    print(f"- {clean_generated}")

Generated names:
- ninohadros
- saurus
- us
- sinornithosaurus
- norasaurus
- phornis
- oceratus
- aisaurus
- eustreptospondylus
- vayosaurus


In [14]:
model.save('dino_name_generator.keras')
print("Model saved as dino_name_generator.keras")

Model saved as dino_name_generator.keras


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)
